In [15]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
import html
import unicodedata

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /Users/mix/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/mix/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
def spell_preprocessor(s):
    s = str(s).lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')
    s = re.sub(r'[^a-z\s]', ' ', s)
    return s

In [17]:
class SpellChecker:
    def __init__(self, word_freqs):
        self.WORDS = word_freqs
        self.N = sum(word_freqs.values())

    def P(self, word):
        return self.WORDS.get(word, 0) / self.N

    def correction(self, word):
        return max(self.candidates(word), key=self.P)

    def candidates(self, word):
        return (self.known([word]) or self.known(self.edits1(word)) or self.known(self.edits2(word)) or [word])

    def known(self, words):
        return set(w for w in words if w in self.WORDS)

    def edits1(self, word):
        letters    = 'abcdefghijklmnopqrstuvwxyz'
        splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
        deletes    = [L + R[1:]               for L, R in splits if R]
        transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R)>1]
        replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
        inserts    = [L + c + R               for L, R in splits for c in letters]
        return set(deletes + transposes + replaces + inserts)

    def edits2(self, word):
        return (e2 for e1 in self.edits1(word) for e2 in self.edits1(e1))

In [18]:
with open('resource/recipes.pkl', 'rb') as f:
    recipes = pickle.load(f)

In [19]:
recipes["Name"] = recipes["Name"].astype(str).apply(html.unescape)
recipes["Description"] = recipes["Description"].astype(str).apply(html.unescape)
recipes["RecipeIngredientParts"] = recipes["RecipeIngredientParts"].astype(str).apply(html.unescape)
recipes["RecipeInstructions"] = recipes["RecipeInstructions"].astype(str).apply(html.unescape)
recipes["Keywords"] = recipes["Keywords"].astype(str).apply(html.unescape)

recipes['Corpus'] = recipes['Name'] + ' ' + recipes['RecipeIngredientParts'] + ' ' + recipes['RecipeInstructions'] + ' ' + recipes['Description'] + ' ' + recipes['Keywords']

In [20]:
spell_vectorizer = CountVectorizer(preprocessor=spell_preprocessor, min_df=100)
spell_counts = spell_vectorizer.fit_transform(recipes['Corpus'])

In [21]:
vocab_counts = spell_counts.sum(axis=0).A1
vocab_words = spell_vectorizer.get_feature_names_out()
word_frequencies = dict(zip(vocab_words, [int(c) for c in vocab_counts]))

In [22]:
spell_collector = SpellChecker(word_frequencies)

with open('resource/spell_collection.pkl', 'wb') as f:
    pickle.dump(spell_collector, f)

In [27]:
with open('resource/spell_collection.pkl', 'rb') as f:
    spell_collector = pickle.load(f)

query = "blueberriee mufin withh chocolet chipps"
query_words = query.split()
    
corrected_words = []
has_typo = False

for word in query_words:
    corrected = spell_collector.correction(word)
    if corrected != word:
        has_typo = True
    corrected_words.append(corrected)

suggested_query = " ".join(corrected_words)
print(suggested_query)

blueberries muffin with chocolate chips
